# Residual MCTS highway evaluation on Colab

This notebook clones a selected Git ref, creates an isolated virtual environment, runs `eval_mcts.py`, and saves per-episode CSV results plus logs to Google Drive.

`eval_mcts.py` writes one CSV row after every episode and resumes from the next unfinished `episode_id` when the same output file already exists (`seed + episode_id`). Keep `RUN_NAME` fixed across reconnects so the notebook points at the same Drive CSV.

In [1]:
# Mount Google Drive. Colab will prompt you to authorize access.
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Experiment parameters (edit this cell before running).
REPO_URL = "https://github.com/thowell332/RQL-Comparison.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
SUPERVISOR_REPO_URL = "https://github.com/thowell332/state-wise-constrained-policy-shaping.git"  #@param {type:"string"}
SUPERVISOR_REPO_REF = "thowell332/results-galore"  #@param {type:"string"}

ENV_NAME = "highway-ME-basic-AddRightReward-v0"  #@param {type:"string"}
BASE_AGENT_CONFIG = "configs/HighwayEnv/agents/MCTSAgent/baselineMEPreRL.json"  #@param {type:"string"}
PRIOR_MODEL_PATH = "logs/highway-ME-basic-v0_Example_Pretrained.zip"  #@param {type:"string"}
N_EPISODES = 10  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
SAFE_DECIDE = False  #@param {type:"boolean"}

# MCTS fields normally supplied by the agent JSON config.
MCTS_SIMULATIONS = 150  # `episodes` in the config
MCTS_HORIZON = 6
MCTS_TEMPERATURE = None  # Keep None to preserve the base config/default.
USE_DQN_PRIOR = True

# Keep this fixed across Colab reconnects to resume the same episodes.csv.
RUN_NAME = "mcts_3l30v_seed42"  #@param {type:"string"}
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/aaai2027/RQL-Comparison/eval_mcts"  #@param {type:"string"}
RESUME = True  #@param {type:"boolean"}

In [3]:
# Clone the requested ref and create a Colab-compatible evaluation environment.
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/RQL-Comparison")
SUPERVISOR_DIR = Path("/content/scps-dependency")
VENV_DIR = Path("/content/venvs/rql-comparison")

# Headless Colab rendering for pygame / highway-env.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OFFSCREEN_RENDERING"] = "1"

for path in (PROJECT_DIR, SUPERVISOR_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)
subprocess.run(["git", "clone", SUPERVISOR_REPO_URL, str(SUPERVISOR_DIR)], check=True)
subprocess.run(["git", "-C", str(SUPERVISOR_DIR), "checkout", SUPERVISOR_REPO_REF], check=True)


def create_venv(venv_dir: Path) -> Path:
    """Create a Colab-friendly venv that reuses the system site packages (incl. CUDA torch)."""
    venv_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["apt-get", "install", "-y", "python3-venv", f"python{sys.version_info.major}.{sys.version_info.minor}-venv"],
        check=False,
    )
    result = subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(venv_dir)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("stdlib venv failed; falling back to virtualenv")
        print(result.stderr)
        if venv_dir.exists():
            shutil.rmtree(venv_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
        subprocess.run(
            [sys.executable, "-m", "virtualenv", "--system-site-packages", str(venv_dir)],
            check=True,
        )
    return venv_dir / "bin" / "python"


VENV_PYTHON = create_venv(VENV_DIR)
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
# The repository-wide requirements pin Python-3.9-era training packages. These are
# the complete dependencies needed by the highway MCTS evaluation path on current Colab.
EVAL_PACKAGES = [
    # gym's RandomNumberGenerator breaks under NumPy>=1.24 deepcopy/pickle.
    "numpy==1.23.5", "gym==0.23.1", "scipy", "pandas", "cloudpickle",
    "matplotlib", "tensorboard", "tensorboardX", "pygame", "tqdm",
    "rich", "pyyaml",
]
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", *EVAL_PACKAGES], check=True)
# eval_mcts.py imports this package, but it is maintained in the companion repo.
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(SUPERVISOR_DIR)],
    check=True,
)

entry_point = PROJECT_DIR / "eval_mcts.py"
if not entry_point.is_file():
    raise FileNotFoundError(f"Selected Git ref does not contain {entry_point}")
print(f"Project: {PROJECT_DIR}")
print(f"Python:  {VENV_PYTHON}")

stdlib venv failed; falling back to virtualenv
Error: Command '['/content/venvs/rql-comparison/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.

Project: /content/RQL-Comparison
Python:  /content/venvs/rql-comparison/bin/python


In [4]:
# Build the runtime agent config, run eval_mcts.py, and save all output to Drive.
import json
import os

RUN_DIR = Path(DRIVE_RESULTS_ROOT).expanduser() / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = RUN_DIR / "episodes.csv"

def resolve_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else PROJECT_DIR / path

base_config_path = resolve_path(BASE_AGENT_CONFIG)
prior_model_path = resolve_path(PRIOR_MODEL_PATH)
for required_path in (base_config_path, prior_model_path):
    if required_path.is_symlink() and not required_path.exists():
        raise FileNotFoundError(
            f"Required asset is a broken symlink: {required_path}. Commit a real file to the selected Git ref."
        )
    if not required_path.is_file() or required_path.stat().st_size == 0:
        raise FileNotFoundError(
            f"Required asset not found: {required_path}. Commit it to the selected Git ref or use an absolute Drive path."
        )

agent_config = json.loads(base_config_path.read_text())
agent_config["episodes"] = MCTS_SIMULATIONS
agent_config["horizon"] = MCTS_HORIZON
agent_config["DQN_prior"] = int(USE_DQN_PRIOR)
agent_config["model_path"] = str(prior_model_path)
if MCTS_TEMPERATURE is not None:
    agent_config["temperature"] = MCTS_TEMPERATURE

runtime_config_path = RUN_DIR / "agent_config.json"
runtime_config_path.write_text(json.dumps(agent_config, indent=2))
manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "supervisor_repo_url": SUPERVISOR_REPO_URL,
    "supervisor_repo_ref": SUPERVISOR_REPO_REF,
    "env_name": ENV_NAME,
    "base_agent_config": str(base_config_path),
    "prior_model_path": str(prior_model_path),
    "n_episodes": N_EPISODES,
    "seed": SEED,
    "safe_decide": SAFE_DECIDE,
    "mcts_simulations": MCTS_SIMULATIONS,
    "mcts_horizon": MCTS_HORIZON,
    "mcts_temperature": MCTS_TEMPERATURE,
    "use_dqn_prior": USE_DQN_PRIOR,
    "output_csv": str(OUTPUT_CSV),
    "resume": RESUME,
}
(RUN_DIR / "run_parameters.json").write_text(json.dumps(manifest, indent=2))

command = [
    str(VENV_PYTHON), str(PROJECT_DIR / "eval_mcts.py"),
    "--env-name", ENV_NAME,
    "--agent-config", str(runtime_config_path),
    "--n-episodes", str(N_EPISODES),
    "--seed", str(SEED),
    "--output", str(OUTPUT_CSV),
]
if SAFE_DECIDE:
    command.append("--safe-decide")
if not RESUME:
    command.append("--no-resume")

print(f"Output CSV: {OUTPUT_CSV}")
print("$", " ".join(command))
log_mode = "a" if RESUME and OUTPUT_CSV.exists() else "w"
with (RUN_DIR / "evaluation.log").open(log_mode, buffering=1) as log:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print(f"Results saved to: {RUN_DIR}")

Output CSV: /content/drive/MyDrive/aaai2027/RQL-Comparison/eval_mcts/mcts_3l30v_seed42/episodes.csv
$ /content/venvs/rql-comparison/bin/python /content/RQL-Comparison/eval_mcts.py --env-name highway-ME-basic-AddRightReward-v0 --agent-config /content/drive/MyDrive/aaai2027/RQL-Comparison/eval_mcts/mcts_3l30v_seed42/agent_config.json --n-episodes 10 --seed 42 --output /content/drive/MyDrive/aaai2027/RQL-Comparison/eval_mcts/mcts_3l30v_seed42/episodes.csv
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + 

CalledProcessError: Command '['/content/venvs/rql-comparison/bin/python', '/content/RQL-Comparison/eval_mcts.py', '--env-name', 'highway-ME-basic-AddRightReward-v0', '--agent-config', '/content/drive/MyDrive/aaai2027/RQL-Comparison/eval_mcts/mcts_3l30v_seed42/agent_config.json', '--n-episodes', '10', '--seed', '42', '--output', '/content/drive/MyDrive/aaai2027/RQL-Comparison/eval_mcts/mcts_3l30v_seed42/episodes.csv']' returned non-zero exit status 1.

In [ ]:
# Verify the artifacts persisted to Drive.
for artifact in sorted(path for path in RUN_DIR.rglob("*") if path.is_file()):
    print(f"{artifact.relative_to(RUN_DIR)}  ({artifact.stat().st_size:,} bytes)")